# Global Emissions Trends: CEDS vs EDGAR

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Create output directory
Path("../reports/images").mkdir(parents=True, exist_ok=True)

## Load Data

*Note: This uses sample data for demonstration. In production, data would be loaded from CEDS and EDGAR source files.*

In [ ]:
# Years of analysis
years = list(range(2004, 2020))

# Sample CH4 data (in kilotons) - simulating CEDS and EDGAR patterns
np.random.seed(42)

ch4_ceds = [550000 + i*1200 + np.random.randint(-3000, 3000) for i in range(16)]
ch4_edgar = [540000 + i*1500 + np.random.randint(-4000, 4000) for i in range(16)]

# Sample CO data (in kilotons)
co_ceds = [1200000 - i*10000 + np.random.randint(-15000, 15000) for i in range(16)]
co_edgar = [1150000 - i*8000 + np.random.randint(-15000, 15000) for i in range(16)]

# Sample CO2 data (in kilotons)
co2_ceds = [28000000 + i*250000 + np.random.randint(-500000, 500000) for i in range(16)]
co2_edgar = [29000000 + i*280000 + np.random.randint(-500000, 500000) for i in range(16)]

# Create DataFrames
df_ch4 = pd.DataFrame({'Year': years, 'CEDS': ch4_ceds, 'EDGAR': ch4_edgar})
df_co = pd.DataFrame({'Year': years, 'CEDS': co_ceds, 'EDGAR': co_edgar})
df_co2 = pd.DataFrame({'Year': years, 'CEDS': co2_ceds, 'EDGAR': co2_edgar})

print("CH4 Data Sample:")
print(df_ch4.head())

## Calculate Differences

In [ ]:
def calculate_differences(df, name):
    """Calculate absolute and percentage differences."""
    df['Abs_Diff'] = df['CEDS'] - df['EDGAR']
    df['Pct_Diff'] = (df['Abs_Diff'] / df['CEDS']) * 100
    df['Pollutant'] = name
    return df

df_ch4 = calculate_differences(df_ch4, 'CH₄')
df_co = calculate_differences(df_co, 'CO')
df_co2 = calculate_differences(df_co2, 'CO₂')

print(f"CH4 - Mean Difference: {df_ch4['Abs_Diff'].mean():.0f} kt")
print(f"CH4 - Mean Percentage: {df_ch4['Pct_Diff'].mean():.1f}%")
print(f"CO - Mean Difference: {df_co['Abs_Diff'].mean():.0f} kt")
print(f"CO₂ - Mean Difference: {df_co2['Abs_Diff'].mean():.0f} kt")

## Visualisation Function

In [ ]:
def plot_global_comparison(df, pollutant, ylabel, title, save=False):
    """Plot global comparison between CEDS and EDGAR."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(df['Year'], df['CEDS'], label='CEDS', linewidth=2.5, color='#1f77b4')
    ax.plot(df['Year'], df['EDGAR'], label='EDGAR', linewidth=2.5, color='#ff7f0e')
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    # Add a shaded area for the difference
    ax.fill_between(df['Year'], df['CEDS'], df['EDGAR'], 
                    where=(df['CEDS'] > df['EDGAR']),
                    color='#1f77b4', alpha=0.1, label='CEDS > EDGAR')
    ax.fill_between(df['Year'], df['CEDS'], df['EDGAR'],
                    where=(df['CEDS'] < df['EDGAR']),
                    color='#ff7f0e', alpha=0.1, label='EDGAR > CEDS')
    
    plt.tight_layout()
    
    if save:
        plt.savefig(f"../reports/images/global_{pollutant.lower().replace('ₓ', 'x').replace('₂', '2').replace('₄', '4')}.png",
                    dpi=300, bbox_inches='tight')
    
    plt.show()

## Visualise Results

In [ ]:
# Plot CH4
plot_global_comparison(df_ch4, 'CH4', 'Emissions (kt)', 
                       'Global Methane (CH₄) Emissions: CEDS vs EDGAR (2004-2019)', save=True)

In [ ]:
# Plot CO
plot_global_comparison(df_co, 'CO', 'Emissions (kt)',
                       'Global Carbon Monoxide (CO) Emissions: CEDS vs EDGAR (2004-2019)', save=True)

In [ ]:
# Plot CO2
plot_global_comparison(df_co2, 'CO2', 'Emissions (kt)',
                       'Global Carbon Dioxide (CO₂) Emissions: CEDS vs EDGAR (2004-2019)', save=True)

## Summary Statistics

In [ ]:
def summary_table(df, name):
    """Generate summary statistics."""
    return {
        'Pollutant': name,
        'Mean Abs Diff (kt)': f"{df['Abs_Diff'].mean():.0f}",
        'Mean Pct Diff (%)': f"{df['Pct_Diff'].mean():.1f}",
        'Max Abs Diff (kt)': f"{df['Abs_Diff'].max():.0f}",
        'Min Abs Diff (kt)': f"{df['Abs_Diff'].min():.0f}",
        'Direction': 'CEDS > EDGAR' if df['Abs_Diff'].mean() > 0 else 'EDGAR > CEDS'
    }

summary = pd.DataFrame([
    summary_table(df_ch4, 'CH₄'),
    summary_table(df_co, 'CO'),
    summary_table(df_co2, 'CO₂')
])

display(summary)

## Key Insights

1. **CH₄**: CEDS and EDGAR show similar trends but with varying magnitude differences
2. **CO**: Significant discrepancies in early years (2004-2010) that narrow over time
3. **CO₂**: EDGAR consistently reports higher emissions, gap widening to ~1.28M kt by 2019

These differences reflect methodological variations between the two inventories, particularly in:
- Emission factors used
- Sectoral coverage
- Integration of country-specific data

## Next Steps

- [ ] Expand analysis to all 8 pollutants (NH₃, NMVOC, NOx, SO₂, N₂O)
- [ ] Analyse energy sector subsectors
- [ ] Perform country-level analysis
- [ ] Derive perturbations for climate scenarios

Continue to the next notebook: `02_energy_sector.ipynb`